# PG-TFIDF computation: manual and automated workflows

PG-TFIDF converts highly variable gene expression into a cell-by-functional-cluster representation. The sections below cover the matrix calculation, the released command-line scripts and the `cell-cluster-marker-tfidf-analysis` workflow.

The manual implementation supports small-data verification. Publication runs should use the released scripts or the prescribed workflow without changing the formulas.

## 1. Parameters

| Parameter | Role | Standard value |
|:--|:--|:--|
| `top_k` | Top expressed HVGs used for functional-cluster mapping and PG-TFIDF | 100 |
| `top_gene_exp` | Short auxiliary per-cell gene list; it does not define PG-TFIDF | 10 |
| `hvg_n_top_genes` | HVGs retained when the skill performs upstream HVG selection | 3,000 |

The public `build_cell_pg_tfidf.py` expects a prepared HVG h5ad and does not expose an internal `--hvg` option. The skill's general builder can inspect a full-gene matrix and apply `--hvg --hvg_n_top_genes 3000` when required.

## 2. Environment and repository paths

Create the released environment from the repository root before running the notebook or command-line scripts.

In [ ]:
# Run these commands in a terminal, not inside the notebook kernel.
environment_commands = [
    "conda env create -f cell-pg-tfidf-gene-functional-clusters/environment.yml",
    "conda activate cell-pg-tfidf",
]
print("\n".join(environment_commands))

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

def locate_repository(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "cell-pg-tfidf-gene-functional-clusters").is_dir():
            return candidate
    raise FileNotFoundError("Run the notebook from this repository or one of its subdirectories.")

ROOT = locate_repository()
MODULE = ROOT / "cell-pg-tfidf-gene-functional-clusters"
EXAMPLE = MODULE / "examples" / "data" / "zheng68k"
OUTPUT = ROOT / "outputs" / "tutorial_pg_tfidf"

H5AD = EXAMPLE / "zheng68k_hvg_32cells_128genes.h5ad"
GENE_CLUSTERS = EXAMPLE / "state_gene_clusters_128.xlsx"
CELL_IDS = EXAMPLE / "cell_id_table_32cells.xlsx"
CELL_LABELS = EXAMPLE / "cell_clusters_32cells.xlsx"
RECORDED_TFIDF = EXAMPLE / "zheng68k_tfidf_32cells.npy"
TOP_K = 100

## 3. Input checks

The bundled Zheng68K subset contains 128 genes and supports code-path checks; it is not intended to reproduce manuscript-scale results. A full run should use the prepared 3,000-HVG h5ad or apply the HVG audit and preparation workflow to a full-gene matrix.

In [ ]:
required = [H5AD, GENE_CLUSTERS, CELL_IDS, CELL_LABELS]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing tutorial inputs:\n" + "\n".join(missing))

adata = sc.read_h5ad(H5AD)
gene_clusters = pd.read_excel(GENE_CLUSTERS)

expected_cluster_columns = {"gene_id", "cluster", "representative_gene"}
if not expected_cluster_columns.issubset(gene_clusters.columns):
    raise ValueError(f"Required gene-cluster columns are missing: {expected_cluster_columns}")
if not adata.obs_names.is_unique or not adata.var_names.is_unique:
    raise ValueError("Cell IDs and gene IDs must be unique.")

print({"cells": adata.n_obs, "genes": adata.n_vars, "gene_clusters": gene_clusters['cluster'].nunique()})

## 4. Manual PG-TFIDF calculation

### 4.1 Select top expressed genes and map them to functional clusters

The released script ranks genes independently within each cell. Genes without a matching entry in the functional-cluster table are excluded after selection. The vocabulary is built from all cluster IDs in the supplied cluster table so that dimensions remain consistent across datasets using the same table.

In [ ]:
gene_to_cluster = {}
for _, record in gene_clusters.iterrows():
    gene_id = str(record["gene_id"]).strip()
    if not gene_id or pd.isna(record["cluster"]):
        continue
    gene_to_cluster[gene_id] = int(record["cluster"])
cluster_ids = sorted(set(gene_to_cluster.values()))
cluster_to_col = {cluster_id: col for col, cluster_id in enumerate(cluster_ids)}
gene_names = np.asarray(adata.var_names.astype(str))
top_k = min(TOP_K, adata.n_vars)

mapped_clusters_per_cell = []
for row_index in range(adata.n_obs):
    row = adata.X[row_index]
    values = row.toarray().ravel() if sparse.issparse(row) else np.asarray(row).ravel()
    top_indices = np.argsort(values)[-top_k:][::-1]
    top_genes = gene_names[top_indices]
    mapped_clusters_per_cell.append([gene_to_cluster[g] for g in top_genes if g in gene_to_cluster])

print("Mapped cluster entries for the first cell:", len(mapped_clusters_per_cell[0]))

### 4.2 Build the cell-by-cluster count matrix

A functional cluster can occur more than once in one cell because several top genes may map to the same cluster. Those repeated assignments define the count $n_{ic}$.

In [ ]:
rows, cols, data = [], [], []
for cell_index, assignments in enumerate(mapped_clusters_per_cell):
    if not assignments:
        continue
    columns = np.asarray([cluster_to_col[c] for c in assignments], dtype=np.int64)
    unique_columns, counts = np.unique(columns, return_counts=True)
    rows.extend([cell_index] * len(unique_columns))
    cols.extend(unique_columns.tolist())
    data.extend(counts.astype(np.float32).tolist())

counts = sparse.csr_matrix(
    (np.asarray(data, dtype=np.float32), (rows, cols)),
    shape=(adata.n_obs, len(cluster_ids)),
    dtype=np.float32,
)
print("Count matrix shape and nonzero entries:", counts.shape, counts.nnz)

### 4.3 Calculate TF, smoothed IDF and row-L2-normalized PG-TFIDF

In [ ]:
EPS = 1e-12
row_sum = np.asarray(counts.sum(axis=1)).ravel().astype(np.float32)
tf = sparse.diags(1.0 / np.maximum(row_sum, EPS)) @ counts

document_frequency = np.diff(counts.tocsc().indptr).astype(np.int64)
idf = (
    np.log((1.0 + counts.shape[0]) / (1.0 + document_frequency.astype(np.float32))) + 1.0
).astype(np.float32)
tfidf = tf.multiply(idf)

row_l2 = np.sqrt(np.asarray(tfidf.multiply(tfidf).sum(axis=1)).ravel().astype(np.float32))
pg_tfidf = (sparse.diags(1.0 / np.maximum(row_l2, EPS)) @ tfidf).tocsr()

print("PG-TFIDF shape:", pg_tfidf.shape)
print("Nonempty-row L2 norms:", np.unique(np.round(np.sqrt(np.asarray(pg_tfidf.multiply(pg_tfidf).sum(axis=1)).ravel()), 6)))

### 4.4 Optional equivalence check

The bundled TF-IDF array is a recorded workflow input. Compare it only when its row order and vocabulary match the h5ad and cluster table used above. A shape mismatch should be investigated rather than coerced.

In [ ]:
if RECORDED_TFIDF.exists():
    recorded = np.load(RECORDED_TFIDF)
    manual = pg_tfidf.toarray()
    if recorded.shape == manual.shape:
        print("Maximum absolute difference:", np.max(np.abs(recorded - manual)))
    else:
        print("Comparison skipped because shapes differ:", recorded.shape, manual.shape)

## 5. Manual label-conditioned statistics

The downstream script reports four different quantities. They should not be merged conceptually.

- **Frequency:** fraction of all cells with a positive PG-TFIDF value for a functional cluster.
- **Mean TF-IDF:** mean PG-TFIDF value across all cells.
- **Presence:** within-label fraction of cells with a positive value.
- **Specificity:** label mean divided by the unweighted mean of label means, with an added `1e-9` denominator constant.

Presence and specificity are label-conditioned. Frequency and mean TF-IDF describe dataset-wide detectability and magnitude; they do not determine biological direction.

In [ ]:
cell_id_table = pd.read_excel(CELL_IDS)
label_table = pd.read_excel(CELL_LABELS)
if cell_id_table["Cell_ID"].duplicated().any() or label_table["Cell_ID"].duplicated().any():
    raise ValueError("Duplicate Cell_ID values must be resolved before alignment.")
if len(cell_id_table) != pg_tfidf.shape[0]:
    raise ValueError("PG-TFIDF rows do not match the Cell_ID table.")

aligned = cell_id_table[["Cell_ID"]].copy()
aligned["embedding_row"] = np.arange(len(aligned))
aligned = aligned.merge(label_table[["Cell_ID", "Louvain_Cluster"]], on="Cell_ID", how="inner")
matrix = pg_tfidf[aligned["embedding_row"].to_numpy()].toarray()
labels = aligned["Louvain_Cluster"].astype(str).to_numpy()

frequency = (matrix > 0).mean(axis=0)
mean_tfidf = matrix.mean(axis=0)
label_order = sorted(np.unique(labels))
label_profiles = np.vstack([matrix[labels == label].mean(axis=0) for label in label_order])
presence = np.vstack([(matrix[labels == label] > 0).mean(axis=0) for label in label_order])
specificity = label_profiles / (label_profiles.mean(axis=0) + 1e-9)

cluster_columns = [str(cluster_id) for cluster_id in cluster_ids]
specificity_table = pd.DataFrame(specificity, index=label_order, columns=cluster_columns)
presence_table = pd.DataFrame(presence, index=label_order, columns=cluster_columns)
specificity_table.iloc[:, :5]

## 6. Released-script workflow

Use the public scripts for a standard repository run. The construction command below uses a prepared HVG h5ad. The statistics command aligns labels by `Cell_ID`; the PG-TFIDF matrix rows must correspond exactly to the Cell_ID table rows.

In [ ]:
def command_text(parts):
    return subprocess.list2cmdline([str(part) for part in parts])

build_command = [
    "python", MODULE / "scripts" / "build_cell_pg_tfidf.py",
    "--h5ad_file", H5AD,
    "--cluster_xlsx", GENE_CLUSTERS,
    "--cluster_protein_id_col", "gene_id",
    "--cluster_id_col", "cluster",
    "--cluster_rep_id_col", "representative_gene",
    "--top_k", "100",
    "--top_gene_exp", "10",
    "--seed", "42",
    "--save_path", OUTPUT / "pg_tfidf",
]
print(command_text(build_command))

In [ ]:
statistics_command = [
    "python", MODULE / "scripts" / "cell_cluster_tfidf_analysis.py",
    "--embedding_path", OUTPUT / "pg_tfidf" / "top_gene_cell_embedding_tfidf.npy",
    "--tfidf_vocab", OUTPUT / "pg_tfidf" / "tfidf_vocab.json",
    "--cell_id_table", CELL_IDS,
    "--cell_cluster_label", CELL_LABELS,
    "--cell_cluster_cols", "Louvain_Cluster,Kmeans_Cluster",
    "--cluster_label_name", "zheng68k_tutorial",
    "--save_path", OUTPUT / "tfidf_statistics",
]
print(command_text(statistics_command))

### Expected primary outputs

| Stage | Output | Meaning |
|:--|:--|:--|
| Construction | `top_gene_cell_embedding_tfidf.npy` | Dense row-L2-normalized PG-TFIDF |
| Construction | `top_gene_cell_embedding_tfidf_sparse.npz` | Sparse PG-TFIDF |
| Construction | `tfidf_vocab.json` | Functional-cluster column order |
| Construction | `tfidf_idf.npy`, `tfidf_df.npy` | IDF and document-frequency vectors |
| Statistics | `cluster_statistics_*.xlsx` | Dataset-wide frequency, mean and variance |
| Statistics | `label_cluster_profiles_*.xlsx` | Label mean PG-TFIDF |
| Statistics | `cluster_presence_*.xlsx` | Within-label positive fraction |
| Statistics | `cluster_specificity_*.xlsx` | Relative label enrichment |

## 7. Skill installation and invocation

A skill is installed as a complete directory containing `SKILL.md` and its supporting scripts and references. The format follows the [OpenAI Skills documentation](https://developers.openai.com/api/docs/guides/tools-skills).

| Skill | Repository source | Use in the supplied cases |
|:--|:--|:--|
| `cell-cluster-marker-tfidf-analysis` | `agent-based-pg-tfidf-analysis-framework/skills/cell-cluster-marker-tfidf-analysis/` | PG-TFIDF prerequisites, label-conditioned statistics, RNA/ADT markers and integrated interpretation |
| `RNAseq-data-preprocessed` | `agent-based-pg-tfidf-analysis-framework/skills/RNAseq-data-preprocessed/` | Input conversion and Cell_ID-aligned RNA/ADT preparation in the SKCM case |
| `protein-gene-cell-emb-constructing` | not bundled in this repository | Upstream embedding or TF-IDF prerequisite generation when required by the SKCM case |

### Install the bundled skills

In Codex, request installation from the GitHub repository with `$skill-installer`:

```text
Use $skill-installer to install cell-cluster-marker-tfidf-analysis from https://github.com/luozeyu1024/PG-TFIDF/tree/main/agent-based-pg-tfidf-analysis-framework/skills/cell-cluster-marker-tfidf-analysis
Use $skill-installer to install RNAseq-data-preprocessed from https://github.com/luozeyu1024/PG-TFIDF/tree/main/agent-based-pg-tfidf-analysis-framework/skills/RNAseq-data-preprocessed
```

For a manual local installation, copy each complete skill directory into the personal Codex skills directory (typically `%USERPROFILE%\.codex\skills\` on Windows), keeping `SKILL.md`, `scripts/` and `references/` together. Start a new Codex task after installation so that skill discovery is refreshed. Do not copy only `SKILL.md`.

Install the bundled runtime separately from the repository root:

```powershell
conda env create -f agent-based-pg-tfidf-analysis-framework/environment.yml
conda activate agent-pg-tfidf
```

### Confirm and call the analysis skill

The canonical name is read from the `name` field in `SKILL.md`: `cell-cluster-marker-tfidf-analysis`. Invoke it explicitly with:

```text
Use $cell-cluster-marker-tfidf-analysis to audit the supplied inputs and run the PG-TFIDF workflow.
```

The Dataset81 case calls `cell-cluster-marker-tfidf-analysis`. The SKCM case calls `RNAseq-data-preprocessed`, `protein-gene-cell-emb-constructing` and `cell-cluster-marker-tfidf-analysis` in sequence according to input availability.

## 8. Automated skill workflow

The `cell-cluster-marker-tfidf-analysis` skill automates input auditing and preset-script orchestration. It does not replace the PG-TFIDF formula. Its standard contract is:

1. audit the h5ad HVG status;
2. use `top_k=100` and retain `top_gene_exp=10`;
3. use the general preset builder, adding `--hvg --hvg_n_top_genes 3000` only when the audit requires upstream HVG selection;
4. calculate specificity only from `top_gene_cell_embedding_tfidf.npy` and `tfidf_vocab.json`;
5. verify Cell_ID overlap and row order before label-conditioned analysis; and
6. report frequency, mean TF-IDF, presence and specificity as distinct quantities.

The `cell-cluster-marker-tfidf-analysis` skill must be installed separately.

### Dataset-specific workflow prompts

Dataset81 and SKCM_GSE115978_aPD1 workflow prompts are provided in [`pg-tfidf-agent-skill-prompts.md`](pg-tfidf-agent-skill-prompts.md).

In [ ]:
PROMPT_CASES = ROOT / "tutorials" / "pg-tfidf-agent-skill-prompts.md"
if not PROMPT_CASES.exists():
    raise FileNotFoundError(PROMPT_CASES)
print(PROMPT_CASES.read_text(encoding="utf-8"))

### Equivalent skill-preset command sequence

The installed skill uses the following script order. Set `CELL_CLUSTER_TFIDF_SKILL_ROOT` to the installed skill directory. The first command creates a heuristic HVG audit report. Choose one construction command according to that report.

In [ ]:
SKILL_ROOT = Path(os.environ.get("CELL_CLUSTER_TFIDF_SKILL_ROOT", "path/to/cell_cluster_marker_tfidf_analysis"))
SKILL_SCRIPTS = SKILL_ROOT / "scripts"
FULL_OR_UNCERTAIN_H5AD = Path("path/to/full_or_uncertain_input.h5ad")
VERIFIED_HVG_H5AD = Path("path/to/verified_hvg_input.h5ad")
SKILL_OUTPUT = Path("outputs/skill_pg_tfidf")

hvg_audit_command = [
    "python", SKILL_SCRIPTS / "check_hvg_status.py",
    "--h5ad", FULL_OR_UNCERTAIN_H5AD,
    "--json", SKILL_OUTPUT / "hvg_status.json",
]

full_matrix_build_command = [
    "python", SKILL_SCRIPTS / "top_gene_cell_embedding_cluster_based_cell_cluster_vector_general.py",
    "--h5ad_file", FULL_OR_UNCERTAIN_H5AD,
    "--cluster_xlsx", GENE_CLUSTERS,
    "--cluster_protein_id_col", "gene_id",
    "--cluster_id_col", "cluster",
    "--cluster_rep_id_col", "representative_gene",
    "--top_k", "100", "--top_gene_exp", "10",
    "--hvg", "--hvg_n_top_genes", "3000",
    "--save_path", SKILL_OUTPUT / "prerequisites",
]

verified_hvg_build_command = [
    "python", SKILL_SCRIPTS / "top_gene_cell_embedding_cluster_based_cell_cluster_vector_general.py",
    "--h5ad_file", VERIFIED_HVG_H5AD,
    "--cluster_xlsx", GENE_CLUSTERS,
    "--cluster_protein_id_col", "gene_id",
    "--cluster_id_col", "cluster",
    "--cluster_rep_id_col", "representative_gene",
    "--top_k", "100", "--top_gene_exp", "10",
    "--save_path", SKILL_OUTPUT / "prerequisites",
]

for title, command in [
    ("HVG audit", hvg_audit_command),
    ("Full-gene or unverified input", full_matrix_build_command),
    ("Verified HVG input", verified_hvg_build_command),
]:
    print(f"\n# {title}\n{command_text(command)}")

In [ ]:
skill_statistics_command = [
    "python", SKILL_SCRIPTS / "cell_tfidf_cluster_analysis.py",
    "--embedding_path", SKILL_OUTPUT / "prerequisites" / "top_gene_cell_embedding_tfidf.npy",
    "--tfidf_vocab", SKILL_OUTPUT / "prerequisites" / "tfidf_vocab.json",
    "--cell_id_table", CELL_IDS,
    "--cell_cluster_label", CELL_LABELS,
    "--cell_cluster_cols", "Louvain_Cluster,Kmeans_Cluster",
    "--save_path", SKILL_OUTPUT / "statistics",
]
print(command_text(skill_statistics_command))

## 9. Validation checklist

Before interpreting the results, confirm all of the following:

- the h5ad was a verified HVG subset or the preset 3,000-HVG route was applied;
- `top_k=100` controlled PG-TFIDF mapping and `top_gene_exp=10` controlled only the auxiliary list;
- PG-TFIDF rows follow `adata.obs_names`;
- the Cell_ID table contains one row per PG-TFIDF row in the same order;
- the vocabulary column count matches the matrix column count;
- nonempty PG-TFIDF rows have an L2 norm close to one;
- specificity was calculated from the released PG-TFIDF matrix, not from raw expression;
- label tables contain unique Cell_ID values and overlap counts are reported; and
- the functional annotations correspond to the same gene-cluster table used for mapping.

In [ ]:
def validate_pg_tfidf(matrix, vocabulary, cell_ids):
    matrix = sparse.csr_matrix(matrix) if not sparse.issparse(matrix) else matrix.tocsr()
    if matrix.shape[0] != len(cell_ids):
        raise ValueError("Matrix rows and Cell_ID rows differ.")
    if matrix.shape[1] != len(vocabulary):
        raise ValueError("Matrix columns and vocabulary entries differ.")
    if pd.Series(cell_ids).astype(str).duplicated().any():
        raise ValueError("Cell_ID values are not unique.")
    norms = np.sqrt(np.asarray(matrix.multiply(matrix).sum(axis=1)).ravel())
    nonempty = norms > 0
    if nonempty.any() and not np.allclose(norms[nonempty], 1.0, atol=1e-5):
        raise ValueError("Nonempty PG-TFIDF rows are not row-L2-normalized.")
    return {
        "shape": matrix.shape,
        "nonempty_cells": int(nonempty.sum()),
        "empty_cells": int((~nonempty).sum()),
    }

validate_pg_tfidf(pg_tfidf, cluster_ids, cell_id_table["Cell_ID"].astype(str))